In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path="/content/drive/MyDrive/New folder/Copy of data.csv"

In [ ]:
import pandas as pd
df = pd.read_csv(path)
print("Shape:", df.shape)

Shape: (34176, 1275)


In [ ]:
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()[:5]}...")  # Show first 5 column names
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")

# Look at the data
print("\nFirst 5 rows:")
print(df.head())

# Check the last column (assuming it's labels)
print(f"\nLast column name: {df.columns[-1]}")
print(f"Unique values in last column: {df.iloc[:, -1].unique()}")
print(f"Number of unique labels: {len(df.iloc[:, -1].unique())}")

# Count of each label
print(f"\nLabel distribution:")
print(df.iloc[:, -1].value_counts())

Dataset Information:
Shape: (34176, 1275)
Columns: ['-288.14898681640625', '-300.1120910644531', '-313.1480712890625', '-314.75970458984375', '-315.8346252441406']...
Total rows: 34176
Total columns: 1275

First 5 rows:
   -288.14898681640625  -300.1120910644531  -313.1480712890625  \
0          -556.491821         -546.531921         -549.918579   
1          -628.486267         -622.119751         -622.733765   
2          -570.658813         -564.907715         -563.779236   
3          -579.377136         -575.636536         -573.867920   
4          -595.857666         -580.376892         -582.433777   

   -314.75970458984375  -315.8346252441406  -314.03369140625  \
0          -459.302795         -189.527100        -92.891487   
1          -622.907471         -619.837158       -619.732117   
2          -304.617157         -132.021194        -74.879395   
3          -573.770508         -574.695862       -421.262390   
4          -584.296204         -584.109131       -583.679382   

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Separate features and labels
X = df.iloc[:, :-1].values  # All columns except last (1274 features)
y = df.iloc[:, -1].values   # Last column (labels)

print("Data shapes:")
print(f"Features (X): {X.shape}")
print(f"Labels (y): {y.shape}")

# Convert text labels to numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nLabel mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{i}: {label}")

# Normalize the features (important for neural networks)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeature scaling done!")
print(f"Original feature range: {X.min():.2f} to {X.max():.2f}")
print(f"Scaled feature range: {X_scaled.min():.2f} to {X_scaled.max():.2f}")

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Keep same proportion of each class
)

print(f"\nData split:")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")
print(f"Classes: {len(label_encoder.classes_)}")

# Check if data is ready
print(f"\n✓ Data is ready for training!")

Data shapes:
Features (X): (34176, 1274)
Labels (y): (34176,)

Label mapping:
0: backward
1: down
2: forward
3: go
4: left
5: no
6: right
7: stop
8: up
9: yes

Feature scaling done!
Original feature range: -1005.56 to 326.57
Scaled feature range: -9.94 to 8.41

Data split:
Training samples: 27340
Test samples: 6836
Features: 1274
Classes: 10

✓ Data is ready for training!


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Create a simple neural network (easy for hardware implementation)
model = keras.Sequential([
    # Input layer: 1274 features
    layers.Dense(256, activation='relu', input_shape=(1274,), name='hidden1'),
    layers.Dropout(0.3),

    # Hidden layer
    layers.Dense(128, activation='relu', name='hidden2'),
    layers.Dropout(0.2),

    # Output layer: 10 classes
    layers.Dense(10, activation='softmax', name='output')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Show model architecture
print("Model Architecture:")
model.summary()

print(f"\nModel layers:")
for i, layer in enumerate(model.layers):
    if hasattr(layer, 'units'):
        print(f"Layer {i}: {layer.name} - {layer.units} units, activation: {layer.activation.__name__}")

print(f"\nModel is ready for training!")
print(f"Input: 1274 features → Hidden1: 256 → Hidden2: 128 → Output: 10 classes")

Model Architecture:


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden1 (Dense)                 │ (None, 256)            │       326,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 360,586 (1.38 MB)

 Trainable params: 360,586 (1.38 MB)

 Non-trainable params: 0 (0.00 B)


Model layers:
Layer 0: hidden1 - 256 units, activation: relu
Layer 2: hidden2 - 128 units, activation: relu
Layer 4: output - 10 units, activation: softmax

Model is ready for training!
Input: 1274 features → Hidden1: 256 → Hidden2: 128 → Output: 10 classes


In [ ]:
print("Starting training...")

history = model.fit(
    X_train, y_train,
    batch_size=64,
    epochs=20,
    validation_data=(X_test, y_test),
    verbose=1
)

print("Training completed!")

Starting training...
Epoch 1/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.4714 - loss: 1.5107 - val_accuracy: 0.7418 - val_loss: 0.7604
Epoch 2/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.7197 - loss: 0.8146 - val_accuracy: 0.7942 - val_loss: 0.6198
Epoch 3/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7743 - loss: 0.6559 - val_accuracy: 0.8104 - val_loss: 0.5554
Epoch 4/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.7950 - loss: 0.6023 - val_accuracy: 0.8309 - val_loss: 0.5095
Epoch 5/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.8108 - loss: 0.5470 - val_accuracy: 0.8325 - val_loss: 0.4976
Epoch 6/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8273 - loss: 0.5005 - val_accuracy: 0.8417 - val_loss: 0.4661
Epoch 7/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.8376 - loss: 0.4773 - val_accuracy: 0.8490 - val_loss: 0.4440
Epoch 8/20
428/428 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8390 - lo

In [ ]:
# Evaluate your trained model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Model test accuracy (float32): {test_acc:.4f}")

214/214 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8644 - loss: 0.4246
Model test accuracy (float32): 0.8670


In [ ]:
!rm -f *.txt *.v


# Step 1: Quantize the Model to INT8

In [ ]:
import numpy as np
import json

def find_optimal_scale(data, target_bits=8):
    """Find optimal scale factor for quantization"""
    if target_bits == 8:
        max_val = 127
        dtype = np.int8
    else:
        max_val = 32767
        dtype = np.int16

    # Use 99.9th percentile instead of max to handle outliers
    max_abs = np.percentile(np.abs(data), 99.9)
    if max_abs == 0:
        return 1.0, dtype

    scale = max_val / max_abs
    return scale, dtype

# Step 1: Calibrate scales using training data
print("Calibrating quantization scales...")

# Find optimal input scale using training data
input_scale, _ = find_optimal_scale(X_train)
print(f"Input scale: {input_scale:.2f}")

# Find optimal scales for each layer
layer_scales = {}
for i in [0, 2, 4]:
    weights, biases = model.layers[i].get_weights()

    weight_scale, _ = find_optimal_scale(weights)
    bias_scale, _ = find_optimal_scale(biases)

    layer_scales[i] = {
        'weight_scale': weight_scale,
        'bias_scale': bias_scale
    }
    print(f"Layer {i}: weight_scale={weight_scale:.2f}, bias_scale={bias_scale:.2f}")

# Step 2: Quantize and save
def quantize_to_int8(data, scale):
    """Quantize data to int8 using specified scale"""
    data_q = np.round(data * scale).astype(np.int8)
    data_q = np.clip(data_q, -127, 127)
    return data_q

for i in [0, 2, 4]:
    weights, biases = model.layers[i].get_weights()

    # Quantize using calibrated scales
    weights_q = quantize_to_int8(weights, layer_scales[i]['weight_scale'])
    biases_q = quantize_to_int8(biases, layer_scales[i]['bias_scale'])

    # Save quantized values
    np.savetxt(f'layer_{i}_weights_calibrated.txt', weights_q, fmt='%d')
    np.savetxt(f'layer_{i}_biases_calibrated.txt', biases_q, fmt='%d')

# Quantize test samples
X_test_10_q = quantize_to_int8(X_test[:10], input_scale)
np.savetxt('X_test_10_calibrated.txt', X_test_10_q, fmt='%d')
np.savetxt('y_test_10.txt', y_test[:10], fmt='%d')

# Save scales for inference (fix for JSON serialization)
scales_data = {
    'input_scale': float(input_scale),
    'layer_scales': {}
}

# Convert all numpy values to Python native types
for layer_idx, layer_scale in layer_scales.items():
    scales_data['layer_scales'][str(layer_idx)] = {
        'weight_scale': float(layer_scale['weight_scale']),
        'bias_scale': float(layer_scale['bias_scale'])
    }

with open('calibrated_scales.json', 'w') as f:
    json.dump(scales_data, f, indent=2)

print("Calibrated quantization completed!")

Calibrating quantization scales...
Input scale: 35.47
Layer 0: weight_scale=425.16, bias_scale=132.99
Layer 2: weight_scale=315.87, bias_scale=467.75
Layer 4: weight_scale=288.87, bias_scale=277.84
Calibrated quantization completed!


# Step 2: Convert INT8 to Properly Scaled Q8.8 Fixed-Point

In [ ]:
import numpy as np
import json

# Load calibrated scales
with open('calibrated_scales.json', 'r') as f:
    scales_data = json.load(f)

# Much more conservative Q8.8 range - use just 1/4 of the available range
MAX_INT_PART = 32  # Use only 1/4 of available range for safety

# Load and calculate max values
X_test_q = np.loadtxt('X_test_10_calibrated.txt', dtype=np.int8)
X_float = X_test_q.astype(np.float32) / scales_data['input_scale']
max_abs_input = np.max(np.abs(X_float))

# More aggressive rescaling
input_rescale = MAX_INT_PART / max_abs_input
print(f"Conservative input rescale factor: {input_rescale:.4f}")

def int8_to_q88_conservative(value_int8, scale, additional_scale=1.0):
    """Convert int8 to very conservative Q8.8 range"""
    value_float = value_int8.astype(np.float32) / scale * additional_scale
    value_q88 = np.round(value_float * 256).astype(np.int16)
    value_q88 = np.clip(value_q88, -8192, 8191)  # Even more conservative clipping
    return value_q88

# Apply conservative rescaling to all values
X_test_q88 = int8_to_q88_conservative(X_test_q, scales_data['input_scale'], input_rescale)
np.savetxt('X_test_10_q88_conservative.txt', X_test_q88, fmt='%d')
print(f"Conservative inputs range: [{X_test_q88.min()}, {X_test_q88.max()}]")

# For layer 0 weights, compensate for input rescaling
for i in [0, 2, 4]:
    weights_q = np.loadtxt(f'layer_{i}_weights_calibrated.txt', dtype=np.int8)
    biases_q = np.loadtxt(f'layer_{i}_biases_calibrated.txt', dtype=np.int8)

    additional_scale = 1.0
    if i == 0:
        additional_scale = 1.0 / input_rescale

    weights_q88 = int8_to_q88_conservative(weights_q, scales_data['layer_scales'][str(i)]['weight_scale'], additional_scale)
    biases_q88 = int8_to_q88_conservative(biases_q, scales_data['layer_scales'][str(i)]['bias_scale'])

    np.savetxt(f'layer_{i}_weights_q88_conservative.txt', weights_q88, fmt='%d')
    np.savetxt(f'layer_{i}_biases_q88_conservative.txt', biases_q88, fmt='%d')

    print(f"Layer {i}:")
    print(f"  Weights range: [{weights_q88.min()}, {weights_q88.max()}]")
    print(f"  Biases range: [{biases_q88.min()}, {biases_q88.max()}]")

# Run the ROM generation code again using these new conservative files

Conservative input rescale factor: 8.9370
Conservative inputs range: [-8192, 8127]
Layer 0:
  Weights range: [-9, 9]
  Biases range: [-227, 243]
Layer 2:
  Weights range: [-103, 103]
  Biases range: [-50, 70]
Layer 4:
  Weights range: [-102, 110]
  Biases range: [-117, 112]


# Step 3: Generate ROM Files for Verilog

In [ ]:
# Generate ROMs with conservative values
import numpy as np

def generate_verilog_rom(module_name, data_file, output_file, is_2d=True):
    """Generate Verilog ROM module for Q8.8 fixed-point data"""

    # Load data
    data = np.loadtxt(data_file, dtype=np.int16)
    if data.ndim == 1:
        data = data.reshape(-1, 1)

    num_rows, num_cols = data.shape
    addr_bits = max(1, (num_rows - 1).bit_length())
    index_bits = max(1, (num_cols - 1).bit_length()) if num_cols > 1 else 1

    with open(output_file, 'w') as f:
        f.write(f"// Q8.8 Fixed-Point ROM (Conservative): {module_name}\n")
        f.write(f"module {module_name} (\n")

        if is_2d and num_cols > 1:
            f.write(f"    input [{addr_bits-1}:0] addr,\n")
            f.write(f"    input [{index_bits-1}:0] index,\n")
            f.write( "    output reg signed [15:0] data_out\n")
        else:
            f.write(f"    input [{addr_bits-1}:0] addr,\n")
            f.write( "    output reg signed [15:0] data_out\n")

        f.write(");\n\n")

        f.write("    always @(*) begin\n")

        if is_2d and num_cols > 1:
            f.write("        case(addr)\n")

            for r in range(num_rows):
                f.write(f"            {addr_bits}'d{r}: begin\n")
                f.write("                case(index)\n")

                for c in range(min(20, num_cols)):
                    val = int(data[r][c])
                    hex_val = f"{(val & 0xFFFF):04X}" if val < 0 else f"{val:04X}"
                    f.write(f"                    {index_bits}'d{c}: data_out = 16'h{hex_val};\n")

                # Add some samples from the middle for larger arrays
                if num_cols > 50:
                    for c in [50, 100, 200, 500, 1000]:
                        if c < num_cols:
                            val = int(data[r][c])
                            hex_val = f"{(val & 0xFFFF):04X}" if val < 0 else f"{val:04X}"
                            f.write(f"                    {index_bits}'d{c}: data_out = 16'h{hex_val};\n")

                # Add last few values
                for c in range(max(20, num_cols-5), num_cols):
                    if c < num_cols:
                        val = int(data[r][c])
                        hex_val = f"{(val & 0xFFFF):04X}" if val < 0 else f"{val:04X}"
                        f.write(f"                    {index_bits}'d{c}: data_out = 16'h{hex_val};\n")

                f.write("                    default: data_out = 16'h0000;\n")
                f.write("                endcase\n")
                f.write("            end\n")

            f.write("            default: data_out = 16'h0000;\n")
            f.write("        endcase\n")
        else:
            f.write("        case(addr)\n")
            for r in range(num_rows):
                val = int(data[r][0] if num_cols > 1 else data[r])
                hex_val = f"{(val & 0xFFFF):04X}" if val < 0 else f"{val:04X}"
                f.write(f"            {addr_bits}'d{r}: data_out = 16'h{hex_val};\n")

            f.write("            default: data_out = 16'h0000;\n")
            f.write("        endcase\n")

        f.write("    end\n")
        f.write("endmodule\n")

    print(f"Generated: {output_file}")

# Generate ROM modules with conservative scaled values
print("Generating Verilog ROM modules with conservative scaling...")

# Generate all 7 ROM files
generate_verilog_rom("rom_layer0_weights", "layer_0_weights_q88_conservative.txt", "rom_layer0_weights.v", is_2d=True)
generate_verilog_rom("rom_layer0_biases", "layer_0_biases_q88_conservative.txt", "rom_layer0_biases.v", is_2d=False)
generate_verilog_rom("rom_layer2_weights", "layer_2_weights_q88_conservative.txt", "rom_layer2_weights.v", is_2d=True)
generate_verilog_rom("rom_layer2_biases", "layer_2_biases_q88_conservative.txt", "rom_layer2_biases.v", is_2d=False)
generate_verilog_rom("rom_layer4_weights", "layer_4_weights_q88_conservative.txt", "rom_layer4_weights.v", is_2d=True)
generate_verilog_rom("rom_layer4_biases", "layer_4_biases_q88_conservative.txt", "rom_layer4_biases.v", is_2d=False)
generate_verilog_rom("rom_test_inputs", "X_test_10_q88_conservative.txt", "rom_test_inputs.v", is_2d=True)

print("\nAll conservative ROM modules generated!")

Generating Verilog ROM modules with conservative scaling...
Generated: rom_layer0_weights.v
Generated: rom_layer0_biases.v
Generated: rom_layer2_weights.v
Generated: rom_layer2_biases.v
Generated: rom_layer4_weights.v
Generated: rom_layer4_biases.v
Generated: rom_test_inputs.v

All conservative ROM modules generated!


/tmp/ipython-input-1422584505.py:68: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  val = int(data[r][0] if num_cols > 1 else data[r])


# Testing

In [ ]:
import numpy as np
import json

# Load conservative Q8.8 fixed-point weights and biases
w0_q88 = np.loadtxt('layer_0_weights_q88_conservative.txt', dtype=np.int16)
b0_q88 = np.loadtxt('layer_0_biases_q88_conservative.txt', dtype=np.int16)
w1_q88 = np.loadtxt('layer_2_weights_q88_conservative.txt', dtype=np.int16)
b1_q88 = np.loadtxt('layer_2_biases_q88_conservative.txt', dtype=np.int16)
w2_q88 = np.loadtxt('layer_4_weights_q88_conservative.txt', dtype=np.int16)
b2_q88 = np.loadtxt('layer_4_biases_q88_conservative.txt', dtype=np.int16)

# Load conservative Q8.8 test inputs
X_test_q88 = np.loadtxt('X_test_10_q88_conservative.txt', dtype=np.int16)
y_test_10 = np.loadtxt('y_test_10.txt', dtype=int)

def q88_to_float(q88_value):
    """Convert Q8.8 fixed-point to float for testing"""
    return q88_value.astype(np.float32) / 256.0

def simulate_q88_network(sample_q88):
    """Simulate Q8.8 fixed-point network (same as Verilog would do)"""
    # Layer 0: input -> hidden1
    x = q88_to_float(sample_q88)
    w0 = q88_to_float(w0_q88)
    b0 = q88_to_float(b0_q88)
    z0 = np.dot(x, w0) + b0
    a0 = np.maximum(0, z0)  # ReLU

    # Layer 1: hidden1 -> hidden2
    w1 = q88_to_float(w1_q88)
    b1 = q88_to_float(b1_q88)
    z1 = np.dot(a0, w1) + b1
    a1 = np.maximum(0, z1)  # ReLU

    # Layer 2: hidden2 -> output
    w2 = q88_to_float(w2_q88)
    b2 = q88_to_float(b2_q88)
    z2 = np.dot(a1, w2) + b2

    return z2

# Test conservative Q8.8 model on all 10 samples
correct = 0
print("Predictions from conservatively scaled Q8.8 model:")
print("Sample | True | Predicted | Match? | Raw Logits")
print("------ | ---- | --------- | ------ | ----------")

for i in range(10):
    logits = simulate_q88_network(X_test_q88[i])
    pred = np.argmax(logits)
    match = "✓" if pred == y_test_10[i] else "✗"
    print(f"{i:6d} | {y_test_10[i]:4d} | {pred:9d} | {match:6s} | {np.round(logits, 2)}")
    if pred == y_test_10[i]:
        correct += 1

q88_accuracy = correct / 10
print(f"\nConservatively scaled Q8.8 model accuracy: {q88_accuracy:.2f}")

# Convert logits for sample 2 to Q8.8 integer values for comparison with Verilog
sample2_logits = simulate_q88_network(X_test_q88[2])
sample2_q88 = np.round(sample2_logits * 256).astype(np.int16)
print(f"\nSample 2 logits in Q8.8 format: {sample2_q88}")

# Save expected predictions for Verilog verification
expected_preds = [np.argmax(simulate_q88_network(X_test_q88[i])) for i in range(10)]
np.savetxt('expected_q88_conservative_predictions.txt', expected_preds, fmt='%d')

Predictions from conservatively scaled Q8.8 model:
Sample | True | Predicted | Match? | Raw Logits
------ | ---- | --------- | ------ | ----------
     0 |    5 |         4 | ✗      | [-0.48 -0.36 -0.95  0.01  1.62  0.45  1.09 -1.29  0.05  0.68]
     1 |    3 |         3 | ✓      | [-5.85 -0.59 -6.07  5.04 -2.2   3.54 -4.02 -0.47  4.15 -4.49]
     2 |    2 |         2 | ✓      | [-0.4  -2.87  9.83  1.46  0.69 -2.83 -1.41  4.91  4.07 -4.55]
     3 |    5 |         3 | ✗      | [-3.05  2.71 -3.19  4.62 -3.76  3.89 -2.87  0.37  1.33 -4.46]
     4 |    8 |         8 | ✓      | [ -0.41  -2.42  -6.88  -1.14   0.93  -3.22  -1.94   3.41  12.57 -10.7 ]
     5 |    5 |         5 | ✓      | [-4.25 -3.37 -6.88  1.65  0.77  3.63 -2.84 -1.05  0.27  1.89]
     6 |    5 |         4 | ✗      | [ 0.05 -1.19 -3.7   0.72  3.3   2.99  0.63 -3.02 -0.3   1.74]
     7 |    4 |         4 | ✓      | [ -1.97  -0.99  -0.63  -3.37   9.55   0.23   3.15 -10.16  -6.24   5.14]
     8 |    7 |         5 | ✗      | [-2.

In [ ]:
# Check first few values of sample 2 in our proper file
import numpy as np

# These should be properly scaled values (-20000 to +20000 range)
proper_values = np.loadtxt('X_test_10_q88_conservative.txt', dtype=np.int16)
print("First 5 values from X_test_10_q88_conservative.txt (sample 2):")
print(proper_values[2, :5])

First 5 values from X_test_10_q88_conservative.txt (sample 2):
[ -710 -1097 -1226 -1355 -1484]
